In [1]:
import torch

depth, rows, d, e = 2, 3, 4, 2
column = d * e  
x = torch.randn(depth, rows, column)

# Reshape
x_reshaped = x.view(depth, rows, d, e)

print(x,'\n\n\n',x_reshaped)

tensor([[[ 2.4892,  0.6623,  0.0122, -0.4613,  0.1303, -1.9524,  0.2701,
          -0.1641],
         [ 1.4674,  1.0820, -1.2316, -0.6500, -0.5269,  0.4068, -1.0352,
           0.7177],
         [ 0.2785,  1.7979,  0.9429, -1.1943,  1.6831,  1.2408, -0.5053,
          -1.7293]],

        [[ 0.3844, -1.4321, -0.3281, -1.7117, -0.1599,  1.8871,  0.7891,
          -1.8707],
         [-0.1402,  0.2497, -0.8663, -0.5210,  0.6845, -1.8598, -0.2722,
          -1.8703],
         [ 0.9465,  0.1574, -0.5293, -0.9047, -1.5778, -0.0565, -0.7275,
           0.4132]]]) 


 tensor([[[[ 2.4892,  0.6623],
          [ 0.0122, -0.4613],
          [ 0.1303, -1.9524],
          [ 0.2701, -0.1641]],

         [[ 1.4674,  1.0820],
          [-1.2316, -0.6500],
          [-0.5269,  0.4068],
          [-1.0352,  0.7177]],

         [[ 0.2785,  1.7979],
          [ 0.9429, -1.1943],
          [ 1.6831,  1.2408],
          [-0.5053, -1.7293]]],


        [[[ 0.3844, -1.4321],
          [-0.3281, -1.7117],
      

In [2]:
import torch

rows, d, e = 3, 4, 2
column = d * e  
x = torch.randn(rows, column)

# Reshape
x_reshaped = x.view(rows, d, e)

print(x,'\n\n\n',x_reshaped)

tensor([[ 0.2829, -0.4879, -0.7191,  2.0264, -0.3402,  1.6154, -0.1205,  0.0735],
        [-0.5820,  1.1902, -0.9243,  0.6518, -0.9703, -0.3433, -2.0741,  0.8199],
        [ 0.2435, -1.2322, -1.8310,  0.1840, -1.5033,  2.0262,  0.3944,  0.4001]]) 


 tensor([[[ 0.2829, -0.4879],
         [-0.7191,  2.0264],
         [-0.3402,  1.6154],
         [-0.1205,  0.0735]],

        [[-0.5820,  1.1902],
         [-0.9243,  0.6518],
         [-0.9703, -0.3433],
         [-2.0741,  0.8199]],

        [[ 0.2435, -1.2322],
         [-1.8310,  0.1840],
         [-1.5033,  2.0262],
         [ 0.3944,  0.4001]]])


## Multi headed attention tests

In [3]:
from model import MultiHeadedAttention
import torch.nn as nn
# ============= TEST SUITE =============

def test_basic_functionality():
    """Test 1: Basic functionality and shape consistency"""
    print("Test 1: Basic Functionality")
    print("-" * 30)
    
    # Parameters
    batch_size = 2
    seq_len = 10
    embed_size = 512
    heads = 8
    
    # Create model
    mha = MultiHeadedAttention(embed_size, heads)
    
    # Create random inputs
    x = torch.randn(batch_size, seq_len, embed_size)
    
    # Self-attention (Q=K=V)
    output = mha(x, x, x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Shape match: {x.shape == output.shape}")
    print(f"✅ Test 1 PASSED\n" if x.shape == output.shape else "❌ Test 1 FAILED\n")

def test_different_sequence_lengths():
    """Test 2: Different sequence lengths for cross-attention"""
    print("Test 2: Cross-Attention with Different Sequence Lengths")
    print("-" * 50)
    
    batch_size = 3
    embed_size = 256
    heads = 4
    
    query_len = 5
    key_len = 8
    
    mha = MultiHeadedAttention(embed_size, heads)
    
    query = torch.randn(batch_size, query_len, embed_size)
    key = torch.randn(batch_size, key_len, embed_size)
    value = torch.randn(batch_size, key_len, embed_size)  # Same length as key
    
    output = mha(query, key, value)
    
    print(f"Query shape: {query.shape}")
    print(f"Key shape: {key.shape}")
    print(f"Value shape: {value.shape}")
    print(f"Output shape: {output.shape}")
    
    expected_shape = (batch_size, query_len, embed_size)
    print(f"Expected output shape: {expected_shape}")
    print(f"✅ Test 2 PASSED\n" if output.shape == expected_shape else "❌ Test 2 FAILED\n")

def test_gradient_flow():
    """Test 3: Gradient flow (backpropagation works)"""
    print("Test 3: Gradient Flow")
    print("-" * 20)
    
    embed_size = 128
    heads = 4
    
    mha = MultiHeadedAttention(embed_size, heads)
    x = torch.randn(1, 5, embed_size, requires_grad=True)
    
    output = mha(x, x, x)
    loss = output.sum()
    loss.backward()
    
    # Check if gradients exist
    has_gradients = all(param.grad is not None for param in mha.parameters())
    input_has_grad = x.grad is not None
    
    print(f"Model parameters have gradients: {has_gradients}")
    print(f"Input has gradients: {input_has_grad}")
    print(f"✅ Test 3 PASSED\n" if has_gradients and input_has_grad else "❌ Test 3 FAILED\n")

def test_attention_properties():
    """Test 4: Attention properties (weights sum to 1)"""
    print("Test 4: Attention Properties")
    print("-" * 25)
    
    # Create a simple case to inspect attention weights
    embed_size = 64
    heads = 2
    seq_len = 4
    
    mha = MultiHeadedAttention(embed_size, heads)
    
    # Set deterministic weights for testing
    torch.manual_seed(42)
    x = torch.randn(1, seq_len, embed_size)
    
    # We need to modify the forward pass to return attention weights
    # For now, we'll just check that the output is reasonable
    output = mha(x, x, x)
    
    # Check that output is not NaN or Inf
    is_finite = torch.isfinite(output).all()
    print(f"Output is finite (no NaN/Inf): {is_finite}")
    print(f"Output mean: {output.mean().item():.4f}")
    print(f"Output std: {output.std().item():.4f}")
    print(f"✅ Test 4 PASSED\n" if is_finite else "❌ Test 4 FAILED\n")

def test_edge_cases():
    """Test 5: Edge cases"""
    print("Test 5: Edge Cases")
    print("-" * 15)
    
    # Test with minimum dimensions
    embed_size = 8
    heads = 2
    
    mha = MultiHeadedAttention(embed_size, heads)
    
    # Single token
    x_single = torch.randn(1, 1, embed_size)
    output_single = mha(x_single, x_single, x_single)
    
    # Large sequence
    x_large = torch.randn(1, 100, embed_size)
    output_large = mha(x_large, x_large, x_large)
    
    print(f"Single token input: {x_single.shape} -> {output_single.shape}")
    print(f"Large sequence input: {x_large.shape} -> {output_large.shape}")
    
    single_ok = output_single.shape == x_single.shape
    large_ok = output_large.shape == x_large.shape
    
    print(f"✅ Test 5 PASSED\n" if single_ok and large_ok else "❌ Test 5 FAILED\n")

def test_parameter_count():
    """Test 6: Parameter count verification"""
    print("Test 6: Parameter Count")
    print("-" * 20)
    
    embed_size = 512
    heads = 8
    
    mha = MultiHeadedAttention(embed_size, heads)
    
    total_params = sum(p.numel() for p in mha.parameters())
    
    # Expected: 4 linear layers, each with embed_size x embed_size weights + embed_size biases
    expected_params = 4 * (embed_size * embed_size + embed_size)
    
    print(f"Total parameters: {total_params:,}")
    print(f"Expected parameters: {expected_params:,}")
    print(f"✅ Test 6 PASSED\n" if total_params == expected_params else "❌ Test 6 FAILED\n")

def compare_with_pytorch_mha():
    """Test 7: Compare output patterns with PyTorch's MultiheadAttention"""
    print("Test 7: Comparison with PyTorch MHA (Pattern Check)")
    print("-" * 50)
    
    embed_size = 128
    heads = 4
    seq_len = 6
    batch_size = 2
    
    # Your implementation
    custom_mha = MultiHeadedAttention(embed_size, heads)
    
    # PyTorch's implementation
    pytorch_mha = nn.MultiheadAttention(embed_size, heads, batch_first=True)
    
    # Same input
    x = torch.randn(batch_size, seq_len, embed_size)
    
    # Get outputs
    custom_output = custom_mha(x, x, x)
    pytorch_output, _ = pytorch_mha(x, x, x)
    
    print(f"Custom MHA output shape: {custom_output.shape}")
    print(f"PyTorch MHA output shape: {pytorch_output.shape}")
    print(f"Shape match: {custom_output.shape == pytorch_output.shape}")
    
    # Check if both produce finite values
    custom_finite = torch.isfinite(custom_output).all()
    pytorch_finite = torch.isfinite(pytorch_output).all()
    
    print(f"Custom MHA finite: {custom_finite}")
    print(f"PyTorch MHA finite: {pytorch_finite}")
    print(f"✅ Test 7 PASSED\n" if custom_finite and pytorch_finite else "❌ Test 7 FAILED\n")


In [4]:
print("🔍 Testing MultiHeadedAttention Implementation")
print("=" * 50)

try:
    test_basic_functionality()
    test_different_sequence_lengths()
    test_gradient_flow()
    test_attention_properties()
    test_edge_cases()
    test_parameter_count()
    compare_with_pytorch_mha()
    
    print("🎉 All tests completed! Check individual results above.")
    
except Exception as e:
    print(f"❌ Test suite failed with error: {e}")
    import traceback
    traceback.print_exc()

🔍 Testing MultiHeadedAttention Implementation
Test 1: Basic Functionality
------------------------------
Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Shape match: True
✅ Test 1 PASSED

Test 2: Cross-Attention with Different Sequence Lengths
--------------------------------------------------
Query shape: torch.Size([3, 5, 256])
Key shape: torch.Size([3, 8, 256])
Value shape: torch.Size([3, 8, 256])
Output shape: torch.Size([3, 5, 256])
Expected output shape: (3, 5, 256)
✅ Test 2 PASSED

Test 3: Gradient Flow
--------------------
Model parameters have gradients: True
Input has gradients: True
✅ Test 3 PASSED

Test 4: Attention Properties
-------------------------
Output is finite (no NaN/Inf): True
Output mean: 0.0383
Output std: 0.2357
✅ Test 4 PASSED

Test 5: Edge Cases
---------------
Single token input: torch.Size([1, 1, 8]) -> torch.Size([1, 1, 8])
Large sequence input: torch.Size([1, 100, 8]) -> torch.Size([1, 100, 8])
✅ Test 5 PASSED

Test 6: Parame